# GreeSight — Deep Learning Klasifikasi Sampah (CRISP-DM)

**Dataset:** [Recyclable and Household Waste Classification — Alistair King](https://www.kaggle.com/datasets/alistairking/recyclable-and-household-waste-classification)  
**Model:** MobileNetV2 Transfer Learning + Fine-tuning  
**Framework:** TensorFlow / Keras  
**Tujuan:** Mengklasifikasikan 30 jenis sampah dari gambar untuk mendukung fitur AI Waste Scanner GreeSight.

> Jalankan di Google Colab dengan GPU (Runtime → Change runtime type → T4 GPU). Notebook mengunduh dataset melalui `kagglehub`. Jika Kaggle meminta autentikasi, ikuti petunjuk autentikasi Kaggle pada cell download. Seluruh hasil metrik adalah hasil aktual dari eksekusi notebook, bukan angka contoh.

## Alur CRISP-DM
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

## 1. Business Understanding

Masalah yang diangkat adalah pengguna sering kesulitan mengenali jenis sampah dan menentukan langkah pemilahan yang tepat. GreeSight dikembangkan sebagai aplikasi edukasi yang menerima gambar melalui kamera atau galeri, lalu memprediksi jenis sampah.

**Tujuan pemodelan:** membangun model klasifikasi citra multikelas dengan 30 label sesuai struktur dataset. Prediksi model menjadi dasar untuk menampilkan panduan pemilahan. Model ini mengidentifikasi objek/kelas pada dataset, bukan mendeteksi banyak objek sekaligus (object detection). Jika beberapa objek berada dalam satu foto, keluaran softmax tetap berupa distribusi probabilitas kelas untuk keseluruhan gambar.

**Kriteria evaluasi:** accuracy, precision, recall, macro-F1, confusion matrix, serta kestabilan kurva loss/accuracy. Dataset dibagi train 70%, validation 15%, test 15% dengan stratifikasi label dan random seed tetap.

## 2. Data Understanding — instalasi dan pengambilan dataset

In [ ]:
!pip -q install -U kagglehub tensorflow scikit-learn pandas matplotlib seaborn pillow

In [ ]:
import os, json, random, shutil, zipfile, math, sys, platform
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support)
from PIL import Image

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 15
EPOCHS_FINE = 10
AUTOTUNE = tf.data.AUTOTUNE
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
import kagglehub
DATASET_PATH = kagglehub.dataset_download('alistairking/recyclable-and-household-waste-classification')
print('Dataset path:', DATASET_PATH)

# Temukan direktori yang langsung berisi folder kelas (mis. aerosol_cans, cardboard_boxes, ...)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def find_class_root(dataset_path):
    candidates=[]
    for p in Path(dataset_path).rglob('*'):
        if not p.is_dir(): continue
        sub=[x for x in p.iterdir() if x.is_dir()]
        if len(sub) >= 5:
            image_count=sum(1 for x in p.rglob('*') if x.is_file() and x.suffix.lower() in IMAGE_EXTS)
            if image_count: candidates.append((image_count,p))
    if not candidates: raise FileNotFoundError('Folder kelas tidak ditemukan. Periksa struktur dataset.')
    return max(candidates,key=lambda x:x[0])[1]

DATA_DIR = find_class_root(DATASET_PATH)
print('Folder kelas:', DATA_DIR)
class_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
print('Jumlah kelas:', len(class_dirs))
print('Nama kelas:', [p.name for p in class_dirs])

### 2.1 Eksplorasi data

Struktur dataset mempunyai 30 folder kelas. Di dalam setiap kelas terdapat konteks `default` (studio) dan `real_world`. Kedua konteks diperlakukan sebagai satu label kelas, tetapi nama konteks disimpan agar evaluasi domain dapat dilakukan terpisah.

In [ ]:
records=[]
for class_dir in class_dirs:
    for p in class_dir.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            context = p.parent.name if p.parent.name.lower() in {'default','real_world'} else 'other'
            records.append({'filepath':str(p),'label':class_dir.name,'context':context})
df = pd.DataFrame(records)
if df.empty: raise ValueError('Tidak ada file gambar yang ditemukan.')
classes = sorted(df.label.unique().tolist())
print('Total gambar:',len(df),'| Jumlah kelas:',len(classes))
print(df.groupby(['label','context']).size().unstack(fill_value=0).head())

plt.figure(figsize=(14,5))
counts=df.label.value_counts().reindex(classes)
sns.barplot(x=counts.index,y=counts.values,color='#919131')
plt.xticks(rotation=75,ha='right'); plt.ylabel('Jumlah gambar'); plt.title('Distribusi gambar per kelas'); plt.tight_layout(); plt.show()

# Contoh gambar dari beberapa kelas
sample=df.groupby('label',group_keys=False).head(1).head(12)
fig,axes=plt.subplots(3,4,figsize=(12,9))
for ax,(_,r) in zip(axes.flat,sample.iterrows()):
    ax.imshow(Image.open(r.filepath).convert('RGB')); ax.set_title(r.label,fontsize=8); ax.axis('off')
for ax in axes.flat[len(sample):]: ax.axis('off')
plt.suptitle('Contoh citra dari dataset'); plt.tight_layout(); plt.show()

## 3. Data Preparation

Tahapan persiapan: validasi file gambar, encode label, split stratified 70:15:15, resize 224×224, preprocessing MobileNetV2, dan augmentasi hanya pada data training. Test set tidak digunakan untuk memilih model.

In [ ]:
# Buang gambar yang tidak dapat dibuka
valid=[]
for r in df.itertuples(index=False):
    try:
        with Image.open(r.filepath) as im: im.verify()
        valid.append({'filepath':r.filepath,'label':r.label,'context':r.context})
    except Exception: pass
df=pd.DataFrame(valid)
classes=sorted(df.label.unique().tolist())
class_to_idx={c:i for i,c in enumerate(classes)}
df['label_id']=df.label.map(class_to_idx).astype('int32')

train_df, temp_df = train_test_split(df,test_size=0.30,random_state=SEED,stratify=df.label_id)
val_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=SEED,stratify=temp_df.label_id)
for name,part in [('Train',train_df),('Validation',val_df),('Test',test_df)]:
    print(f'{name}: {len(part):,} gambar ({len(part)/len(df):.1%}) | {part.label.nunique()} kelas')

# Dataset pipeline: input [0,255], preprocessing MobileNetV2 dilakukan di dalam model.
def decode_image(path,label):
    raw=tf.io.read_file(path)
    image=tf.io.decode_image(raw,channels=3,expand_animations=False)
    image.set_shape([None,None,3])
    image=tf.image.resize(image,IMG_SIZE)
    return image,tf.cast(label,tf.int32)

def make_dataset(part,training=False):
    ds=tf.data.Dataset.from_tensor_slices((part.filepath.values,part.label_id.values))
    if training: ds=ds.shuffle(len(part),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(decode_image,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
train_ds=make_dataset(train_df,True); val_ds=make_dataset(val_df); test_ds=make_dataset(test_df)

# Verifikasi batch
xb,yb=next(iter(train_ds)); print('Batch image:',xb.shape,'| labels:',yb.shape)

## 4. Modeling — MobileNetV2 Transfer Learning

MobileNetV2 dipilih sebagai backbone ringan yang sesuai untuk kebutuhan inferensi aplikasi mobile/web. Tahap pertama membekukan backbone dan melatih classification head. Tahap kedua membuka sebagian layer akhir untuk fine-tuning dengan learning rate kecil. Augmentasi dilakukan secara online pada training saja.

In [ ]:
augmentation=keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.12),
    layers.RandomContrast(0.10),
],name='data_augmentation')
base=keras.applications.MobileNetV2(input_shape=IMG_SIZE+(3,),include_top=False,weights='imagenet')
base.trainable=False
inputs=keras.Input(shape=IMG_SIZE+(3,),name='image')
x=augmentation(inputs)
x=keras.applications.mobilenet_v2.preprocess_input(x)
x=base(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(0.35)(x)
x=layers.Dense(256,activation='relu')(x)
x=layers.Dropout(0.25)(x)
outputs=layers.Dense(len(classes),activation='softmax',name='class_probabilities')(x)
model=keras.Model(inputs,outputs,name='GreeSight_MobileNetV2')
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.summary()

OUT_DIR=Path('/content/greesight_outputs') if Path('/content').exists() else Path('greesight_outputs')
OUT_DIR.mkdir(parents=True,exist_ok=True)
callbacks=[
    keras.callbacks.ModelCheckpoint(OUT_DIR/'best_model.keras',monitor='val_loss',save_best_only=True,verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss',patience=4,restore_best_weights=True,verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.3,patience=2,min_lr=1e-7,verbose=1)
]
history_head=model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_HEAD,callbacks=callbacks)

### 4.1 Fine-tuning

Membuka 40 layer terakhir backbone (kecuali BatchNormalization) untuk penyesuaian fitur yang lebih spesifik terhadap dataset. Learning rate dibuat lebih kecil agar bobot pretrained tidak berubah drastis.

In [ ]:
base.trainable=True
for layer in base.layers[:-40]: layer.trainable=False
for layer in base.layers:
    if isinstance(layer,layers.BatchNormalization): layer.trainable=False
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',metrics=['accuracy'])
history_fine=model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_FINE,callbacks=callbacks)

# Gabungkan histori dua tahap
hist={}
for key in set(history_head.history)|set(history_fine.history):
    hist[key]=history_head.history.get(key,[])+history_fine.history.get(key,[])
metrics_df=pd.DataFrame(hist)
metrics_df.insert(0,'epoch',np.arange(1,len(metrics_df)+1))
metrics_df.to_csv(OUT_DIR/'training_history.csv',index=False)
metrics_df.tail()

## 5. Evaluation

Evaluasi final dilakukan pada test set yang tidak digunakan saat training. Selain metrik agregat, evaluasi menampilkan metrik tiap kelas, confusion matrix, kesalahan prediksi, dan performa berdasarkan konteks gambar `default`/`real_world`. Error trend dihitung sebagai `1 - accuracy` pada training dan validation.

In [ ]:
# Learning curves: accuracy, loss, dan error trend
fig,axs=plt.subplots(1,3,figsize=(17,4))
axs[0].plot(metrics_df.epoch,metrics_df.accuracy,label='Train'); axs[0].plot(metrics_df.epoch,metrics_df.val_accuracy,label='Validation')
axs[0].set_title('Accuracy per Epoch'); axs[0].set_xlabel('Epoch'); axs[0].set_ylabel('Accuracy'); axs[0].legend(); axs[0].grid(alpha=.25)
axs[1].plot(metrics_df.epoch,metrics_df.loss,label='Train'); axs[1].plot(metrics_df.epoch,metrics_df.val_loss,label='Validation')
axs[1].set_title('Loss per Epoch'); axs[1].set_xlabel('Epoch'); axs[1].set_ylabel('Loss'); axs[1].legend(); axs[1].grid(alpha=.25)
axs[2].plot(metrics_df.epoch,1-metrics_df.accuracy,label='Train error'); axs[2].plot(metrics_df.epoch,1-metrics_df.val_accuracy,label='Validation error')
axs[2].set_title('Error Trend (1 - Accuracy)'); axs[2].set_xlabel('Epoch'); axs[2].set_ylabel('Error rate'); axs[2].legend(); axs[2].grid(alpha=.25)
plt.tight_layout(); plt.savefig(OUT_DIR/'learning_curves.png',dpi=180,bbox_inches='tight'); plt.show()

# Load best checkpoint if available
best_path=OUT_DIR/'best_model.keras'
if best_path.exists(): model=keras.models.load_model(best_path)
model.evaluate(test_ds,verbose=1)
y_true=np.concatenate([y.numpy() for _,y in test_ds])
probabilities=model.predict(test_ds,verbose=1)
y_pred=np.argmax(probabilities,axis=1)
acc=accuracy_score(y_true,y_pred)
precision,recall,f1,_=precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
print(f'Test accuracy : {acc:.4f}')
print(f'Macro precision: {precision:.4f} | Macro recall: {recall:.4f} | Macro F1: {f1:.4f}')
report=classification_report(y_true,y_pred,labels=np.arange(len(classes)),target_names=classes,output_dict=True,zero_division=0)
report_df=pd.DataFrame(report).transpose(); report_df.to_csv(OUT_DIR/'classification_report.csv')
print(classification_report(y_true,y_pred,target_names=classes,zero_division=0))

cm=confusion_matrix(y_true,y_pred,labels=np.arange(len(classes)))
plt.figure(figsize=(16,14)); sns.heatmap(cm,cmap='YlGn',xticklabels=classes,yticklabels=classes)
plt.title('Confusion Matrix — Test Set'); plt.xlabel('Predicted label'); plt.ylabel('True label')
plt.xticks(rotation=90,fontsize=7); plt.yticks(fontsize=7); plt.tight_layout()
plt.savefig(OUT_DIR/'confusion_matrix.png',dpi=180,bbox_inches='tight'); plt.show()

# Per-class F1 visualization
per_class=report_df.loc[classes,['precision','recall','f1-score']]
per_class.sort_values('f1-score').plot(kind='barh',figsize=(10,11),color=['#ba5972','#919131','#c1a669'])
plt.title('Precision, Recall, dan F1-score per Kelas'); plt.xlabel('Score'); plt.xlim(0,1); plt.tight_layout()
plt.savefig(OUT_DIR/'per_class_metrics.png',dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# Evaluasi domain: default vs real_world pada test set
# Urutan prediksi test_ds sama dengan urutan test_df karena shuffle=False.
context_test=test_df.context.reset_index(drop=True)
pred_df=test_df[['filepath','label','context']].reset_index(drop=True).copy()
pred_df['true_id']=y_true; pred_df['pred_id']=y_pred
pred_df['predicted_label']=[classes[i] for i in y_pred]
pred_df['confidence']=probabilities.max(axis=1)
pred_df['correct']=pred_df.true_id==pred_df.pred_id
print(pred_df.groupby('context').correct.agg(['count','mean']))
context_metrics=pred_df.groupby('context').correct.mean().rename('accuracy').reset_index()
context_metrics.to_csv(OUT_DIR/'context_evaluation.csv',index=False)

# Sampel kesalahan prediksi untuk analisis
errors=pred_df[~pred_df.correct].sort_values('confidence',ascending=False)
errors.head(100).to_csv(OUT_DIR/'misclassified_samples.csv',index=False)
print('Jumlah prediksi salah:',len(errors),'dari',len(pred_df))
if len(errors):
    sample_err=errors.head(12)
    fig,axes=plt.subplots(3,4,figsize=(13,10))
    for ax,(_,r) in zip(axes.flat,sample_err.iterrows()):
        ax.imshow(Image.open(r.filepath).convert('RGB'))
        ax.set_title(f"True: {r.label}\nPred: {r.predicted_label} ({r.confidence:.0%})",fontsize=8)
        ax.axis('off')
    for ax in axes.flat[len(sample_err):]: ax.axis('off')
    plt.suptitle('Contoh Kesalahan Prediksi'); plt.tight_layout(); plt.savefig(OUT_DIR/'misclassified_examples.png',dpi=160); plt.show()

## 6. Deployment — ekspor model dan artefak

Artefak deployment: `best_model.keras`, `labels.json`, `model_metadata.json`, dan laporan evaluasi. Backend FastAPI membaca model Keras dan menyediakan endpoint `/predict`. Notebook juga membuat ZIP data olahan yang berisi manifest split, label, histori training, laporan evaluasi, serta (opsional) salinan gambar ke folder train/validation/test.

**Catatan ukuran:** ZIP yang menyertakan seluruh gambar dapat berukuran besar. Default `EXPORT_IMAGES_TO_ZIP=False` membuat ZIP ringan dengan manifest dan metadata. Ubah menjadi `True` jika memang diperlukan salinan fisik gambar yang sudah dibagi.

In [ ]:
# Simpan model, label, metadata, dan hasil evaluasi
model.save(OUT_DIR/'greesight_model.keras')
with open(OUT_DIR/'labels.json','w',encoding='utf-8') as f: json.dump(classes,f,ensure_ascii=False,indent=2)
metadata={
    'project':'GreeSight','architecture':'MobileNetV2 Transfer Learning + Fine-tuning',
    'input_size':list(IMG_SIZE),'num_classes':len(classes),'class_names':classes,
    'preprocessing':'keras.applications.mobilenet_v2.preprocess_input inside model',
    'seed':SEED,'split':{'train':0.70,'validation':0.15,'test':0.15},
    'test_metrics':{'accuracy':float(acc),'macro_precision':float(precision),'macro_recall':float(recall),'macro_f1':float(f1)}
}
with open(OUT_DIR/'model_metadata.json','w',encoding='utf-8') as f: json.dump(metadata,f,ensure_ascii=False,indent=2)

# Manifest data olahan: path relatif, label, split, konteks
train_out=train_df.assign(split='train'); val_out=val_df.assign(split='validation'); test_out=test_df.assign(split='test')
manifest=pd.concat([train_out,val_out,test_out],ignore_index=True)
manifest['relative_path']=manifest.filepath.map(lambda x: str(Path(x).relative_to(DATA_DIR)))
manifest[['relative_path','label','label_id','context','split']].to_csv(OUT_DIR/'processed_data_manifest.csv',index=False)

EXPORT_IMAGES_TO_ZIP=False  # True = sertakan semua gambar hasil split (ukuran ZIP besar)
if EXPORT_IMAGES_TO_ZIP:
    image_root=OUT_DIR/'processed_images'
    for row in manifest.itertuples(index=False):
        dest=image_root/row.split/row.label/Path(row.filepath).name
        dest.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(row.filepath,dest)

# Rekap distribusi split per kelas
split_summary=manifest.groupby(['split','label']).size().rename('image_count').reset_index()
split_summary.to_csv(OUT_DIR/'split_class_distribution.csv',index=False)
zip_path=OUT_DIR/'greesight_processed_data.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for file in OUT_DIR.rglob('*'):
        if file.is_file() and file != zip_path and 'processed_images' not in str(file):
            z.write(file,file.relative_to(OUT_DIR))
    if EXPORT_IMAGES_TO_ZIP:
        for file in (OUT_DIR/'processed_images').rglob('*'):
            if file.is_file(): z.write(file,file.relative_to(OUT_DIR))
print('ZIP data olahan:',zip_path,'| size MB:',round(zip_path.stat().st_size/1024**2,2))
print('Isi output:',[p.name for p in OUT_DIR.iterdir()])

## 7. Uji prediksi satu gambar

Pilih satu file gambar untuk melihat top-3 probabilitas kelas. Confidence merupakan probabilitas softmax model dan bukan jaminan kepastian objek.

In [ ]:
def predict_image(image_path, top_k=3):
    img=tf.keras.utils.load_img(image_path,target_size=IMG_SIZE)
    arr=tf.keras.utils.img_to_array(img)[None,...]
    probs=model.predict(arr,verbose=0)[0]
    indices=np.argsort(probs)[::-1][:top_k]
    return [{'label':classes[i],'confidence':float(probs[i])} for i in indices]

print(predict_image(test_df.iloc[0].filepath))

## Catatan interpretasi dan batasan
- Model mengklasifikasikan satu gambar ke salah satu dari 30 kelas dataset, bukan melakukan object detection/segmentasi untuk banyak objek.
- Dataset tidak memberikan aturan daur ulang lokal. Panduan pemilahan pada website merupakan konten edukasi dummy/generik yang perlu disesuaikan dengan kebijakan daerah atau bank sampah setempat.
- Performa pada foto lapangan dapat berbeda dari hasil test. Perhatikan evaluasi terpisah `default` dan `real_world`.
- Simpan `greesight_model.keras`, `labels.json`, dan `model_metadata.json` ke folder `backend/` untuk menjalankan website.